# 📊 Análisis Exploratorio y Modelado de Datos de Conectividad Digital en Colombia

El acceso a Internet es un factor fundamental para el desarrollo social, educativo y económico de un país. Sin embargo, la disponibilidad del servicio no es uniforme entre las distintas regiones, lo que genera brechas digitales que afectan la calidad de vida de la población.

En Colombia, los servicios de Internet fijo presentan variaciones según el departamento, municipio, proveedor, tecnología utilizada y segmento de usuarios. Analizar estos patrones permite entender cómo se distribuye la conectividad en el territorio nacional y cómo ha evolucionado a lo largo del tiempo.

⚠️ **Alcance del proyecto**  
El presente proyecto se enfoca específicamente en el análisis de la conectividad digital en los departamentos de **Antioquia** y **Risaralda**, permitiendo un estudio más detallado y contextualizado de estas regiones.

---

## 🎯 Objetivo del Proyecto

Diseñar una base de datos que permita **almacenar, organizar y analizar** la información relacionada con los accesos a Internet fijo en Colombia, facilitando consultas sobre:

✅ Distribución geográfica del servicio  
✅ Evolución temporal del acceso  
✅ Participación de proveedores  
✅ Uso de tecnologías de conexión  
✅ Comportamiento de los distintos segmentos de usuarios  

# 🧩 Identificación de Entidades, Atributos y Relaciones

El modelo de datos se construyó a partir de las características del dataset original.

---
# 🗄️ Entidad Principal: **Accesos_Internet**

Representa el registro del número de accesos a Internet para una combinación específica de ubicación, proveedor, tecnología y periodo.

### 🔹 Atributos
- **id_acceso** (PK)  
- anio  
- trimestre  
- id_municipio (FK)  
- id_proveedor (FK)  
- id_tecnologia (FK)  
- id_segmento (FK)  
- velocidad_bajada  
- velocidad_subida  
- accesos  

### 🔗 Relaciones
- 📍 Se registra en un municipio  
- 🏢 Es proporcionado por un proveedor  
- 🌐 Utiliza una tecnología  
- 👥 Corresponde a un segmento de usuarios  
---

## 🗺️ Entidad: **Departamentos**

Representa las divisiones administrativas del país.

### 🔹 Atributos
- **id_departamento** (PK)  
- nombre_departamento  

### 🔗 Relación
- Un departamento contiene varios municipios.

---

## 🏙️ Entidad: **Municipios**

Representa las divisiones territoriales dentro de cada departamento.

### 🔹 Atributos
- **id_municipio** (PK)  
- nombre_municipio  
- id_departamento (FK)  

### 🔗 Relaciones
- Pertenece a un departamento  
- Puede tener múltiples registros de accesos a Internet

---

## 🏢 Entidad: **Proveedores**

Representa las empresas que prestan el servicio de Internet.

### 🔹 Atributos
- **id_proveedor** (PK)  
- nombre_proveedor  

### 🔗 Relación
- Un proveedor puede ofrecer servicios en múltiples municipios y periodos

---

## 🌐 Entidad: **Tecnologías**

Representa el tipo de infraestructura utilizada para proveer el servicio.

### 🔹 Atributos
- **id_tecnologia** (PK)  
- nombre_tecnologia  

### 🔗 Relación
- Una tecnología puede estar presente en múltiples registros de accesos

---

## 👥 Entidad: **Segmentos**

Representa el tipo de cliente al que se presta el servicio.

### 🔹 Atributos
- **id_segmento** (PK)  
- nombre_segmento  

### 🔗 Relación
- Un segmento puede aparecer en múltiples registros de accesos

---


In [1]:
 #Librerias
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import csv
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import gdown
!pip install streamlit pyngrok

# Configuración para ver todas las columnas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Carga de datos desde Drive externo
file_id = "1TaD6JL2Pz9j5U0M3rg59djlBcC75TWED"
url = f"https://drive.google.com/uc?id={file_id}"
gdown.download(url, "Internet_Fijo_Accesos.csv", quiet=False)

# Asignacion de datos CSV a Dataset
df=pd.read_csv("/content/Internet_Fijo_Accesos.csv", sep=",")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 77.4 MB/s eta 0:00:00


Downloading...
From (original): https://drive.google.com/uc?id=1TaD6JL2Pz9j5U0M3rg59djlBcC75TWED
From (redirected): https://drive.google.com/uc?id=1TaD6JL2Pz9j5U0M3rg59djlBcC75TWED&confirm=t&uuid=ad7f8fa9-61f4-4638-83d0-7820636240fd
To: /content/Internet_Fijo_Accesos.csv
100%|██████████| 403M/403M [00:06<00:00, 58.9MB/s]


In [2]:
# Construcción de dataset limitado a los departamentos de alcance
df_paisa = df[
    (df["DEPARTAMENTO"].str.strip().str.upper() == "ANTIOQUIA") |
    (df["DEPARTAMENTO"].str.strip().str.upper() == "RISARALDA")
]

# Convertir espacios o strings vacíos en NaN
df_paisa = df_paisa.replace(r'^\s*$', np.nan, regex=True)

# Eliminar filas que tengan cualquier NaN
df_paisa = df_paisa.dropna()

In [3]:
# Preparacion de datasets para tablas independientes(DF dimensión)
df_departamentos = (
    df_paisa[["COD_DEPARTAMENTO", "DEPARTAMENTO"]]
    .drop_duplicates()
    .sort_values("COD_DEPARTAMENTO")
    .reset_index(drop=True)
)
df_municipios = (
    df_paisa[["COD_MUNICIPIO", "MUNICIPIO", "COD_DEPARTAMENTO"]]
    .drop_duplicates()
    .sort_values("COD_MUNICIPIO")
    .reset_index(drop=True) )
df_municipios["MUNICIPIO"] = (
    df_municipios["MUNICIPIO"]
    .str.normalize('NFKD')
    .str.encode('ascii', errors='ignore')
    .str.decode('utf-8')
)
df_proveedores = df_paisa[["PROVEEDOR"]].drop_duplicates()
df_proveedores["PROVEEDOR"] = (
    df_proveedores["PROVEEDOR"]
    .str.normalize('NFKD')
    .str.encode('ascii', errors='ignore')
    .str.decode('utf-8')
)
df_proveedores.insert(0, "id_proveedor", df_proveedores.index + 1)
df_tecnologia = df_paisa[["TECNOLOGIA"]].drop_duplicates()
df_tecnologia["TECNOLOGIA"] = (
    df_tecnologia["TECNOLOGIA"]
    .str.normalize('NFKD')
    .str.encode('ascii', errors='ignore')
    .str.decode('utf-8')
)
df_tecnologia.insert(0, "id_tecnologia", df_tecnologia.index + 1)
df_segmentos = df_paisa[["SEGMENTO"]].drop_duplicates()
df_segmentos["SEGMENTO"] = (
    df_segmentos["SEGMENTO"]
    .str.normalize('NFKD')
    .str.encode('ascii', errors='ignore')
    .str.decode('utf-8')
)
df_segmentos.insert(0, "id_segmento", df_segmentos.index + 1)

In [4]:
# Exportación de datasets independientes a archivos CSV
df_departamentos.to_csv("departamentos.csv", index=False, encoding="utf-8-sig")
df_municipios.to_csv("municipios.csv", index=False, encoding="utf-8-sig")
df_proveedores.to_csv("proveedores.csv", index=False, encoding="utf-8-sig")
df_tecnologia.to_csv("tecnologias.csv", index=False, encoding="utf-8-sig")
df_segmentos.to_csv("segmentos.csv", index=False, encoding="utf-8-sig")

In [37]:
# Preparacion de dataset principal
df_accesos = df_paisa.merge(
    df_proveedores,
    on="PROVEEDOR",
    how="left"
)
df_accesos = df_accesos.merge(
    df_tecnologia,
    on="TECNOLOGIA",
    how="left"
)
df_accesos = df_accesos.merge(
    df_segmentos,
    on="SEGMENTO",
    how="left"
)

df_accesos = df_accesos.dropna(subset=["id_proveedor", "id_tecnologia"])

df_accesos = df_accesos.rename(columns={
    "AÑO": "anio",
    "TRIMESTRE": "trimestre",
    "No DE ACCESOS": "accesos",
    "COD_MUNICIPIO": "id_municipio",
    "COD_DEPARTAMENTO": "id_departamento"
})

df_accesos = df_accesos[[
    "anio",
    "trimestre",
    "id_departamento",
    "id_municipio",
    "id_proveedor",
    "id_tecnologia",
    "id_segmento",
    "VELOCIDAD_BAJADA",
    "VELOCIDAD_SUBIDA",
    "accesos"
]]
df_accesos["id_municipio"] = df_accesos["id_municipio"].astype("int64")
df_accesos["id_departamento"] = df_accesos["id_departamento"].astype("int64")
df_accesos["id_proveedor"] = df_accesos["id_proveedor"].astype("int64")
df_accesos["id_tecnologia"] = df_accesos["id_tecnologia"].astype("int64")
df_accesos["VELOCIDAD_BAJADA"] = (
    df_accesos["VELOCIDAD_BAJADA"]
    .str.replace(",", ".", regex=False)
    .astype(float)
)

df_accesos["VELOCIDAD_SUBIDA"] = (
    df_accesos["VELOCIDAD_SUBIDA"]
    .str.replace(",", ".", regex=False)
    .astype(float)
)
df_accesos["VELOCIDAD_BAJADA"] = df_accesos["VELOCIDAD_BAJADA"].round().astype(int)
df_accesos["VELOCIDAD_SUBIDA"] = df_accesos["VELOCIDAD_SUBIDA"].round().astype(int)
df_accesos.to_csv("accesos.csv", index=False, encoding="utf-8-sig")

In [16]:
# Librerias para dashboard
import streamlit as st

# Consultas para elaboración de dashboard
df_q1 = (
    df_accesos
    .groupby("id_departamento", as_index=False)["accesos"]
    .sum()
    .merge(df_departamentos, left_on="id_departamento", right_on="COD_DEPARTAMENTO")
    .sort_values("accesos", ascending=False)
)

df_q2 = (
    df_accesos
    .groupby("id_municipio", as_index=False)["accesos"]
    .sum()
    .merge(df_municipios, left_on="id_municipio", right_on="COD_MUNICIPIO")
    .merge(df_departamentos, left_on="COD_DEPARTAMENTO", right_on="COD_DEPARTAMENTO")
    .sort_values("accesos", ascending=False)
    .head(10)
)

df_q3 = (
    df_accesos
    .groupby("id_tecnologia", as_index=False)["accesos"]
    .sum()
    .merge(df_tecnologia, on="id_tecnologia")
    .sort_values("accesos", ascending=False)
)

df_q4 = (
    df_accesos
    .groupby("id_segmento", as_index=False)
    .agg({
        "VELOCIDAD_BAJADA":"mean",
        "VELOCIDAD_SUBIDA":"mean"
    })
    .merge(df_segmentos, on="id_segmento")
)

df_q4["VELOCIDAD_BAJADA"] = df_q4["VELOCIDAD_BAJADA"].round(0).astype(int)
df_q4["VELOCIDAD_SUBIDA"] = df_q4["VELOCIDAD_SUBIDA"].round(0).astype(int)

promedio_global = (
    df_accesos
    .groupby("id_proveedor")["accesos"]
    .sum()
    .mean()
)

df_q5 = (
    df_accesos
    .groupby("id_proveedor", as_index=False)["accesos"]
    .sum()
    .query("accesos > @promedio_global")
    .merge(df_proveedores, on="id_proveedor")
    .sort_values("accesos", ascending=False)
)

df_q1.to_csv("q1_departamentos.csv", index=False)
df_q2.to_csv("q2_municipios.csv", index=False)
df_q3.to_csv("q3_tecnologia.csv", index=False)
df_q4.to_csv("q4_velocidades.csv", index=False)
df_q5.to_csv("q5_proveedores.csv", index=False)

In [32]:
# Construcción de dashboard
%%writefile dashboard.py
st.title("Conectividad Digital en Región Paisa")

# -------- Cargar consultas --------
df1 = pd.read_csv("q1_departamentos.csv")
df2 = pd.read_csv("q2_municipios.csv")
df3 = pd.read_csv("q3_tecnologia.csv")
df4 = pd.read_csv("q4_velocidades.csv")
df5 = pd.read_csv("q5_proveedores.csv")

# -------- Consulta 1 --------
st.header("Accesos por Departamento")
st.dataframe(df1)
fig, ax = plt.subplots()
ax.bar(df1.iloc[:,1], df1["accesos"])
plt.xticks(rotation=45)
st.pyplot(fig)

# -------- Consulta 2 --------
st.header("Top Municipios")
st.dataframe(df2)
fig2, ax2 = plt.subplots()
ax2.barh(df2.iloc[:,1], df2["accesos"])
st.pyplot(fig2)

# -------- Consulta 3 --------
st.header("Accesos por Tecnología")
st.dataframe(df3)
fig3, ax3 = plt.subplots()
ax3.pie(df3["accesos"], labels=df3.iloc[:,1], autopct="%1.0f%%")
st.pyplot(fig3)

# -------- Consulta 4 --------
st.header("Velocidad Promedio por Segmento")
st.dataframe(df4)

# -------- Consulta 5 --------
st.header("Proveedores sobre Promedio Global")
st.dataframe(df5)
fig5, ax5 = plt.subplots()
ax5.bar(df5.iloc[:,1], df5["accesos"])
plt.xticks(rotation=75)
st.pyplot(fig5)

Overwriting dashboard.py
